# R06 · Cookbook-Referenzlabor: Konfiguration, Wartung, Dateien, Fehler, Systemraum, CLI

**NHyDB-0.28.2-Notebook-Reihe · v1 · Referenzteil**

Diese Beispiele basieren auf dem offiziellen
[NHyDB Python Cookbook](https://github.com/nhydb-run4/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (Abschnitte 19–24), das als maßgebliche Referenz dient.
Code und erwartete Ausgaben sind unverändert aus dem Cookbook übernommen, die Begleittexte ins
Deutsche übersetzt; dieselben Beispiele führt auch die NHyDB-Testsuite aus.

* Jede Code-Zelle ist ein **eigenständiges Beispiel** – führe eine Zelle einzeln aus oder nutze *Run All*.
* Jedes Beispiel legt seinen eigenen temporären Datenbankordner an.
* Unter jedem Beispiel steht die im Cookbook abgedruckte Ausgabe, zum Vergleich mit deiner.
* Die Hauptreihe (`../main/`) erklärt die Ideen hinter diesen APIs Schritt für Schritt.

**Inhalt** (11 ausführbare Beispiele)

* 19. Konfiguration und Grenzen
* 20. Wartung: Statistik, Zustand, Prüfung, Reparatur, Kompaktierung
* 21. Datenbankdateien, Sicherung, Wiederherstellung und Verschieben
* 22. Fehler, Sperren und gleichzeitiger Zugriff
* 23. Systemraum R0,u1
* 24. Kommandozeile

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.28.2':
    print('⚠ These examples were written for NHyDB 0.28.2; results may differ.')

nhydb 0.28.2 · file format 0.28.2


## 19. Konfiguration und Grenzen

Einstellungen können aus einer TOML-Datei kommen. Die Engine sucht in dieser Reihenfolge:
Argument `config=`, Umgebungsvariable `NHYDB_CONFIG`,
`nhydb_config.toml` neben der Datenbank, eingebaute Standardwerte. Eine kommentierte
Vorlage liegt im Repository unter `config/nhydb_config.example.toml`.

In [2]:
import os
import tempfile

import nhydb

folder = tempfile.mkdtemp()
with open(os.path.join(folder, "nhydb_config.toml"), "w", encoding="utf-8") as f:
    f.write('[database]\ndb_max_size_bytes = 300000\ndurability = "full"\n\n[lock]\ntimeout_ms = 2000\n')

path = os.path.join(folder, "configured.nhydb")
with nhydb.create(path) as db:                       # the file next to the database is found
    print(os.path.basename(db.config_source), db.durability, db.max_db_size_bytes)
    db.create_realm(1, "r")
    try:
        db.put_points(1, [(u, 1, b"x" * 1000, "bytes") for u in range(1, 400)])
    except nhydb.DatabaseSizeLimitError:
        print("size limit reached")

nhydb_config.toml full 300000
size limit reached


*Erwartete Ausgabe (aus dem Cookbook):*

```text
nhydb_config.toml full 300000
size limit reached
```

| Abschnitt / Schlüssel | Bedeutung |
|---|---|
| `[database] db_max_size_bytes` | Größengrenze für **neue** Datenbanken (Standard 50 GiB, mindestens 64 KiB) |
| `[database] durability` | `"fsync"` oder `"full"` (Aliasse `"default"`/`"normal"`, `"fullfsync"`) |
| `[math] realm_auto_limit` | Realm-Budget für neue MATH-Layouts (Standard 4096) |
| `[math] math_max_intermediate_bits` | Bit-Grenze von MATH-Ausdrücken, wird in jedes neue MATH-Layout eingefroren |
| `[lock] enabled`, `timeout_ms` | Sperrdatei und Wartezeit |
| `[paths] layout_path`, `key_path` | Pfadvorlagen, `{db}` = Name der Datenbankdatei, `{db_path_hash}` = Hash des absoluten Datenbankpfads (für gleichnamige Datenbanken mit gemeinsamem Schlüsselordner) |
| `[cache] block_cache_bytes` | Budget des Block-Caches pro geöffnetem Objekt (Standard 64 MiB, LRU; `0` = kein Cache); gecacht werden Blöcke bis max(8 MiB, Budget/8) |
| `[cache] vindex_max_marks`, `vindex_max_indexes` | Budget der gecachten v-Indizes großer einzelner u-Gruppen (in Sprungmarken) und ihre Anzahl |

Explizite Argumente haben Vorrang vor der Datei (`lock=`/`lock_timeout_ms=` überschreiben
die `[lock]`-Schlüssel einzeln). In der Datenbank gespeicherte Werte (die Größengrenze,
das Realm-Budget und die Bit-Grenze eines MATH-Layouts) werden beim Öffnen nie
überschrieben. Unbekannte Schlüssel sind Fehler. Die Größengrenze einer bestehenden
Datenbank änderst du mit `set_max_db_size`:

In [3]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "limit.nhydb")
with nhydb.create(path) as db:
    db.set_max_db_size(10 * 1024**2, commit=True)     # 10 MiB
with nhydb.open(path) as db:
    print(db.max_db_size_bytes)

10485760


*Erwartete Ausgabe (aus dem Cookbook):*

```text
10485760
```

**Block-Cache.** Jeder Block wird pro geöffnetem Objekt einmal vollständig gelesen und per
CRC geprüft; danach bleibt er entweder im Block-Cache oder wird, wenn er nicht
hineinpasst, später nur in den benötigten Teilen gelesen. `stats()["block_cache"]`
zeigt das Budget, den größten cachebaren Block, die gecachten und geprüften
Blöcke sowie die Zähler `hits`, `loads`, `partial_reads` und `evictions`:

In [4]:
import os
import tempfile

import nhydb

folder = tempfile.mkdtemp()
with open(os.path.join(folder, "nhydb_config.toml"), "w", encoding="utf-8") as f:
    f.write("[cache]\nblock_cache_bytes = 16777216\n")      # 16 MiB block cache

path = os.path.join(folder, "cached.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "x", "type": "u32"}])
    db.create_classic_layout(1, 1, "c", shape_id=1)
    db.put_dense_batch(1, 1, [[u] for u in range(1, 10_001)], commit=True)

with nhydb.open(path, readonly=True) as db:
    for u in (1, 5_000, 10_000):
        db.get(1, u)
    bc = db.stats()["block_cache"]
    print(bc["budget_bytes"], bc["max_item_bytes"])
    print(bc["loads"], bc["hits"], bc["cached_blocks"], bc["verified_blocks"])

16777216 8388608
1 2 1 1


*Erwartete Ausgabe (aus dem Cookbook):*

```text
16777216 8388608
1 2 1 1
```

## 20. Wartung: Statistik, Zustand, Prüfung, Reparatur, Kompaktierung

In [5]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "maint.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "x", "type": "u32"}])
    db.create_classic_layout(1, 1, "c", shape_id=1)
    for u in range(1, 201):
        db.put(1, u, [u], commit=True)                # many small commits -> many blocks

    st = db.stats()
    print(st["format"], st["storage"]["physical_dense_records"], st["file_bytes"] > 0)
    h = db.health(deep=True)
    print(h["status"], [(c["name"], c["status"]) for c in h["checks"] if c["status"] != "ok"])
    offset = db.checkpoint()                          # writes a CHECKPOINT block and commits
    print(offset == db.checkpoint_offset, db.health()["status"])
    v = db.verify(deep=True)
    print(v["ok"], v["error_count"])

print(nhydb.NHyDB.inspect_tail(path)["clean"])
res = nhydb.NHyDB.compact_file(path, make_backup=False)   # rewrite the live state compactly
print(res["ok"], res["records_written"], res["after_bytes"] < res["before_bytes"])
with nhydb.open(path) as db:
    print(db.get(1, 200)["x"])

0.28.2 200 True
warning [('checkpoint', 'warning')]
True healthy
True 0
True
True 200 True
200


*Erwartete Ausgabe (aus dem Cookbook):*

```text
0.28.2 200 True
warning [('checkpoint', 'warning')]
True healthy
True 0
True
True 200 True
200
```

* `verify(deep=False)` prüft die Blockstruktur (`deep=True` zusätzlich die
  Prüfsummen der Nutzdaten), läuft nach einem Schaden weiter und listet jeden beschädigten
  Block auf; veraltete Indizes erscheinen unter `warnings` (sie sind keine Fehler).
* `health(deep=False)` führt die Prüfungen `container`, `layouts`, `spacemap`,
  `layout_routing`, `math_layouts`, `definitions`, `indexes` und
  `checkpoint` aus; Status `healthy`, `warning` oder `unhealthy` (ein veralteter Index
  ist eine `warning`). Eine Datenbank ohne aktuellen Checkpoint meldet eine
  `checkpoint`-Warnung;
  `checkpoint()` schreibt einen (eine diagnostische Momentaufnahme der
  Segmentstruktur; ältere Blöcke werden dadurch nicht ersetzt).
* `stats()` meldet Datei, Format, Realms, Shapes, Layouts, Extensions,
  Indizes und Speicherstatistik, unter `block_cache` außerdem den Block-Cache
  dieses Objekts (Abschnitt 19).
* `compact_file(path, output=None, make_backup=True, ...)` ist statisch und
  verlangt, dass die Datenbank geschlossen ist. An Ort und Stelle behält es die Identität und die
  Begleitdateien und schreibt eine Sicherung `<db>.before_compact_<timestamp>` plus
  `<db>.before_compact_<timestamp>.layout.enc` (außer bei
  `make_backup=False`). Mit `output=` exportiert es in eine **neue** Datenbank
  mit eigenen Begleitdateien. Eine fehlgeschlagene Kompaktierung lässt die Quelle unverändert
  und entfernt nur die Dateien, die sie selbst angelegt hat. Der Speicherbedarf ist durch einen
  Batch (`batch_records`) begrenzt, nicht durch die Größe der Datenbank: Datensätze werden
  als Rohbytes kopiert, einzelne Zellen in Stücken.

**Ein beschädigtes Dateiende reparieren.** `inspect_tail` prüft das Ende der Datei,
ohne sie zu öffnen; `repair_file(path, apply=False)` meldet nur,
`apply=True` schneidet das beschädigte Ende ab (standardmäßig mit einer Sicherung
`<db>.bak_<timestamp>`):

In [6]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "damaged.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.put_cell(1, 1, 1, "kept", type="utf8", commit=True)

with open(path, "ab") as f:
    f.write(b"NHBL" + b"\x07" * 40)                  # simulate an incomplete write

tail = nhydb.NHyDB.inspect_tail(path)
print(tail["clean"], tail["trailing_bad_bytes"])
print(nhydb.NHyDB.repair_file(path)["clean"])        # report only
rep = nhydb.NHyDB.repair_file(path, apply=True, make_backup=False)
print(rep["applied"], nhydb.NHyDB.inspect_tail(path)["clean"])
with nhydb.open(path) as db:
    print(db.get_value(1, 1, 1))

False 44
False
True True
kept


*Erwartete Ausgabe (aus dem Cookbook):*

```text
False 44
False
True True
kept
```

`repair_file` opfert committete Daten nie stillschweigend:

* Liegen intakte committete Blöcke hinter einer beschädigten Stelle, löst es
  `CorruptionError` aus und ändert nichts. `allow_data_loss=True` schneidet sie
  trotzdem ab und listet sie in `lost_blocks` auf.
* Nicht committete Blöcke nach der Commit-Grenze werden verworfen
  (`discarded_uncommitted_blocks`); `keep_uncommitted=True` behält die intakten
  (`kept_uncommitted_blocks`).
* Ist ein Superblock beschädigt und folgen vollständige Blöcke auf die Grenze, musst du
  dich ausdrücklich mit einem dieser beiden Schalter entscheiden.
* Der Probelauf (`apply=False`) meldet die geplante `action` (`none`,
  `repair` oder `abort`) und, falls er ablehnen würde, `plan_error`.
* Die Sicherung besteht aus `<db>.bak_<timestamp>` und
  `<db>.bak_<timestamp>.layout.enc`.

Auch das Öffnen einer Datenbank mit beschädigtem Ende funktioniert – der Writer schneidet
alles nach der Commit-Grenze automatisch ab, solange beide
Superblöcke übereinstimmen. Tun sie das nicht (nur möglich nach einem Strom- oder Systemausfall,
während ein Superblock geschrieben wurde – ein abgebrochener Prozess verursacht das nie),
schlägt das Öffnen fehl, ohne die Datei zu ändern: `nhydb.open(path,
readonly=True)` liest weiterhin den vorherigen committeten Zustand, und
`repair_file(path, apply=True, keep_uncommitted=True)` behält die intakten
Blöcke nach der Grenze (`allow_data_loss=True` verwirft sie).

Um zu einer Sicherung von `repair_file` oder `compact_file` zurückzukehren, rufe
`nhydb.NHyDB.restore_backup(backup_path, path)` auf (Kommandozeile:
`nhydb restore BACKUP DB`). Es prüft, dass die Sicherung zur
Datenbank gehört und sich mit ihrem Schlüssel öffnen lässt, und ersetzt dann die Datendatei und den
Layoutplan; die Schlüsseldatei bleibt, wie sie ist.

## 21. Datenbankdateien, Sicherung, Wiederherstellung und Verschieben

| Datei | Inhalt | Hinweis |
|---|---|---|
| `X.nhydb` | Daten (Append-only-Container) | nicht verschlüsselt |
| `X.nhydb.layout.enc` | Schema (Realms, Shapes, Layouts, Indizes, MATH), verschlüsselt | |
| `X.nhydb.layout.key` | 32-Byte-Schlüssel für die Schemadatei | **ohne ihn lässt sich die Datenbank nicht öffnen**; Unix-Rechte `0600` |
| `X.nhydb.lock` | Sperrdatei, leer | darf gelöscht werden, solange die Datenbank geschlossen ist |

Regeln:

* **Sichere alle drei Dateien gemeinsam** (`.nhydb`, `.layout.enc`,
  `.layout.key`), während die Datenbank **geschlossen** ist. Bewahre eine Kopie der
  Schlüsseldatei an einem sicheren Ort auf; ein verlorener Schlüssel lässt sich nicht wiederherstellen.
* Der Dateiname ist Teil der Datenbankidentität: Eine umbenannte `.nhydb` lässt sich
  nicht öffnen (auch nicht, wenn sich nur Groß- und Kleinschreibung ändern). Stelle Sicherungen
  unter dem **gleichen Dateinamen** wieder her (ein anderer Ordner ist in Ordnung). Eine Kopie
  unter neuem Namen legst du mit `compact_file(path, output=new_path)` an.
* Nur das Schema ist verschlüsselt. Die Daten in `X.nhydb` liegen im Klartext;
  schütze die Dateien mit Dateisystemrechten oder Festplattenverschlüsselung.
* Die Schlüsseldatei wird unter Unix mit den Rechten `0600` angelegt; ist sie
  für Gruppe oder andere lesbar, gibt das Öffnen eine `RuntimeWarning` aus (auch in
  `db.open_warnings`). **Windows:** Die Schlüsseldatei erbt die ACL ihres
  Ordners – NHyDB setzt keine eigene ACL. Lege Schlüsseldateien in einen Ordner, auf den nur du
  zugreifen kannst (zum Beispiel unterhalb von `%USERPROFILE%`), oder lass `key_path` dorthin zeigen.
* Eine Sicherungskopie auf schreibgeschützten Medien lässt sich mit
  `nhydb.open(path, readonly=True)` untersuchen.
* **Datenbanken von NHyDB 0.28.0, 0.27.1 oder älter lassen sich** mit 0.28.2 **nicht öffnen**, auch
  nicht mit `readonly=True`: Das Öffnen scheitert sicher (fail closed) mit `CorruptionError`
  (die Meldung nennt das Format) und ändert nichts. Es gibt keine Migration, und
  `repair_file` kann eine solche Datei nicht umwandeln (sein Probelauf meldet
  `action: "abort"`). Exportiere die Daten mit der Version, die sie geschrieben hat, und importiere sie
  in eine neue Datenbank mit 0.28.2.

In [7]:
import os
import shutil
import tempfile

import nhydb

live = tempfile.mkdtemp()
path = os.path.join(live, "crm.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "crm")
    db.put_cell(1, 1, 1, "customer 1", type="utf8")

# backup: copy the three files of the closed database
backup = tempfile.mkdtemp()
for suffix in ("", ".layout.enc", ".layout.key"):
    shutil.copy2(path + suffix, os.path.join(backup, "crm.nhydb" + suffix))

# restore into another folder, same file name
restored = tempfile.mkdtemp()
for suffix in ("", ".layout.enc", ".layout.key"):
    shutil.copy2(os.path.join(backup, "crm.nhydb" + suffix), os.path.join(restored, "crm.nhydb" + suffix))
with nhydb.open(os.path.join(restored, "crm.nhydb")) as db:
    print(db.get_value(1, 1, 1))

# a renamed file is rejected ...
os.rename(os.path.join(restored, "crm.nhydb"), os.path.join(restored, "renamed.nhydb"))
try:
    nhydb.open(os.path.join(restored, "renamed.nhydb"),
               layout_path=os.path.join(restored, "crm.nhydb.layout.enc"),
               key_path=os.path.join(restored, "crm.nhydb.layout.key"))
except nhydb.CorruptionError:
    print("renamed file rejected")

# ... use compact_file(output=...) for a copy under a new name
copy = os.path.join(tempfile.mkdtemp(), "crm_copy.nhydb")
nhydb.NHyDB.compact_file(path, output=copy)
print(sorted(os.listdir(os.path.dirname(copy))))
with nhydb.open(copy) as db:
    print(db.get_value(1, 1, 1))

customer 1
renamed file rejected
['crm_copy.nhydb', 'crm_copy.nhydb.layout.enc', 'crm_copy.nhydb.layout.key', 'crm_copy.nhydb.lock']
customer 1


*Erwartete Ausgabe (aus dem Cookbook):*

```text
customer 1
renamed file rejected
['crm_copy.nhydb', 'crm_copy.nhydb.layout.enc', 'crm_copy.nhydb.layout.key', 'crm_copy.nhydb.lock']
customer 1
```

**Die Schlüsseldatei an einem anderen Ort aufbewahren.** Übergib `key_path` (und optional
`layout_path`) beim Anlegen und bei jedem Öffnen der Datenbank – oder trage die
Pfadvorlagen in die Konfigurationsdatei ein (Abschnitt 19):

In [8]:
import os
import tempfile

import nhydb

data_dir, key_dir = tempfile.mkdtemp(), tempfile.mkdtemp()
path = os.path.join(data_dir, "secure.nhydb")
key = os.path.join(key_dir, "secure.nhydb.layout.key")

with nhydb.create(path, key_path=key) as db:
    db.create_realm(1, "r")
print(sorted(os.listdir(data_dir)), os.listdir(key_dir))

try:
    nhydb.open(path)                                    # key not found next to the database
except OSError:
    print("key missing -> cannot open")
with nhydb.open(path, key_path=key) as db:
    print(1 in db.realms)

['secure.nhydb', 'secure.nhydb.layout.enc', 'secure.nhydb.lock'] ['secure.nhydb.layout.key']
key missing -> cannot open
True


*Erwartete Ausgabe (aus dem Cookbook):*

```text
['limiter.txt', 'secure.nhydb', 'secure.nhydb.layout.enc', 'secure.nhydb.lock'] ['secure.nhydb.layout.key']
key missing -> cannot open
True
```

## 22. Fehler, Sperren und gleichzeitiger Zugriff

Alle Engine-Fehler leiten sich von `nhydb.NHyDBError` ab, einschließlich der Fehler der
MATH-Ausdrücke (`MathError`), des MATH-Realm-Planers (`LayoutError`)
und der Abfrageschicht (`QueryError`). Nur `IndexConfigError` ist stattdessen ein
`ValueError`.

```text
NHyDBError
├── CorruptionError, SchemaError, StaleIndexError, RealmLimitError, AddressError
├── DatabaseLockedError, DatabaseSizeLimitError
├── ResourceLimitError (reason: cancelled_by_app, cancelled_by_limiter, wait_timeout)
├── MathLayoutError ── MathLayoutNotFound, MappingFrozenError, MathCellConflictError
├── MathError ── MathSyntaxError, MathDomainError
├── LayoutError ── LayoutStateError, RealmBudgetExceeded, StalePlanError, AllocationConflictError
└── QueryError ── QueryPlanError, QueryDomainError, SQLSyntaxError
ValueError ── IndexConfigError
```

Die Engine löst auch eingebaute Exceptions aus:

* `ValueError`, `TypeError`, `KeyError` bei ungültigen Argumenten (zum Beispiel
  `TypeError` für `u=2.7`, `ValueError` für IDs über 2^63 − 1 oder Werte, die
  tiefer als 256 Ebenen verschachtelt sind);
* `OverflowError`, wenn eine ganze Python-Zahl nicht in einen 64-Bit-Parameter passt
  (zum Beispiel `u_window=2**64`), `UnicodeEncodeError` für Text mit einzelnen
  Surrogaten;
* `MemoryError`, wenn ein Ergebnis nicht in den Arbeitsspeicher passt;
* Unterklassen von `OSError` mit `errno` und, soweit bekannt, `filename`:
  `FileNotFoundError`, `FileExistsError`, `PermissionError`,
  `IsADirectoryError` …;
* `RuntimeError`, wenn ein Objekt gleichzeitig von zwei Threads verwendet wird
  oder ein Plugin in eine Datenbank zurückruft, die gerade schreibt.

Beschädigte Dateien lösen immer `CorruptionError` aus.

In [9]:
import nhydb

print(issubclass(nhydb.CorruptionError, nhydb.NHyDBError))
print(issubclass(nhydb.MathSyntaxError, nhydb.NHyDBError), issubclass(nhydb.MathSyntaxError, nhydb.MathError))
print(issubclass(nhydb.SQLSyntaxError, nhydb.QueryError), issubclass(nhydb.IndexConfigError, ValueError))
try:
    nhydb.n_to_dynadron_address(576)
except nhydb.NHyDBError as e:
    print(type(e).__name__)

True
True True
True True
AddressError


*Erwartete Ausgabe (aus dem Cookbook):*

```text
True
True True
True True
AddressError
```

**Ein schreibendes Objekt pro Datei.** Das Öffnen einer Datenbank zum Schreiben setzt eine
exklusive Sperre auf `<db>.lock`. Ein zweites solches Öffnen (im selben oder in
einem anderen Prozess) löst `DatabaseLockedError` aus, außer es wartet mit
`lock_timeout_ms`. Leser, die nicht blockieren dürfen, können mit
`readonly=True` öffnen: keine Sperre, nichts an der Datei wird geändert, Schreibaufrufe und
`commit()` lösen `NHyDBError` aus, und das Objekt sieht den committeten Zustand zum
Zeitpunkt des Öffnens (spätere Commits des Writers werden nicht nachgeladen).

In [10]:
import os
import subprocess
import sys
import tempfile
import textwrap

import nhydb

path = os.path.join(tempfile.mkdtemp(), "shared.nhydb")
nhydb.create(path).close()

with nhydb.open(path) as db:
    try:
        nhydb.open(path)                                # same process, second object
    except nhydb.DatabaseLockedError:
        print("locked (same process)")
    with nhydb.open(path, readonly=True) as reader:     # readers need no lock
        print("read-only:", reader.readonly)

holder = textwrap.dedent("""
    import sys, time, nhydb
    db = nhydb.open(sys.argv[1])
    print("locked", flush=True)
    time.sleep(1.0)
    db.close()
""")
proc = subprocess.Popen([sys.executable, "-c", holder, path], stdout=subprocess.PIPE, text=True)
assert proc.stdout.readline().strip() == "locked"
try:
    nhydb.open(path)
except nhydb.DatabaseLockedError:
    print("locked (other process)")
db = nhydb.open(path, lock_timeout_ms=10_000)           # waits until the other process closes
print("opened after waiting")
db.close()
proc.wait(timeout=30)
proc.stdout.close()

locked (same process)
read-only: True
locked (other process)


opened after waiting


*Erwartete Ausgabe (aus dem Cookbook):*

```text
locked (same process)
read-only: True
locked (other process)
opened after waiting
```

`lock=False` schaltet die Sperrdatei ab. Verwende das nur, wenn du garantieren kannst,
dass kein anderer Prozess die Datenbank gleichzeitig öffnet; mit
`lock=False` wird ein nicht committetes Ende nie abgeschnitten (`DatabaseLockedError`).
Die Sperrdatei folgt nie einem symbolischen Link.

**Threads.** Lange Aufrufe geben die GIL frei, während der Rust-Kern arbeitet: Abfragen
(`query`, `sql`, `sql_numpy`, `explain_query`, `QueryEngine`-Methoden),
`stats`, `health`, `verify`, `read_numpy`, die Batch-Schreibmethoden, `commit`,
`rollback` und die statischen Dateioperationen. Python-Callbacks (Abfrage-Provider,
Layout-Plugins) holen sich die GIL für ihren Aufruf zurück. Kurze Aufrufe
wie `get` und `find_u` behalten die GIL absichtlich (die Übergabe würde
mehr kosten als der Aufruf). Mehrere Threads dürfen gleichzeitig nur lesende Abfragen auf
demselben Objekt ausführen; ein Schreiben, `commit` oder `rollback`, während ein anderer
Aufruf das Objekt hält, scheitert sofort mit `RuntimeError`
("Already borrowed") – es wartet nicht und ändert nichts. Für paralleles
Arbeiten gib jedem Thread sein eigenes Objekt (Leser mit `readonly=True`; vier
Threads mit eigenen Objekten führen SQL-Scans etwa 2,9-mal so schnell aus wie einer) oder
schütze ein gemeinsames Objekt mit einem `threading.Lock`.

## 23. Systemraum R0,u1

Jede Datenbank speichert ihre Identität in den reservierten Zellen `(0, 1, v)`:
Datenbank-ID, Dateinamen, Schema-Generation und -Hash, Erstellungszeit,
Verschlüsselungsverfahren und die festen Konstanten.

In [11]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "sys.nhydb")
with nhydb.create(path) as db:
    meta = db.get_system_metadata()
    print(meta["realm"], meta["u"], sorted(meta["cells"], key=int))
    print(db.get_system_cell(2), db.get_system_cell(8))
    print(db.get_system_cell(9)["realm_span"] == 10**100)

0 1 ['1', '2', '3', '4', '5', '6', '7', '8', '9']
sys.nhydb AES-256-GCM + HMAC-SHA256
True


*Erwartete Ausgabe (aus dem Cookbook):*

```text
0 1 ['1', '2', '3', '4', '5', '6', '7', '8', '9']
sys.nhydb AES-256-GCM + HMAC-SHA256
True
```

## 24. Kommandozeile

Das Paket installiert den Befehl `nhydb` (auch `python -m nhydb`) und das
Abfragewerkzeug `python -m nhydb.query`. `--help` zeigt alle Optionen; `--json`
liefert maschinenlesbare Ausgaben.

```bash
nhydb to-n --realm 1 --u 11 --v 13 --lane NR
nhydb from-n N589
nhydb get shop.nhydb u2,v3 --realm 1 --mode cell
nhydb stats shop.nhydb
nhydb health shop.nhydb --deep
nhydb verify shop.nhydb
python -m nhydb.query shop.nhydb --sql "SELECT COUNT(*) AS n FROM orders" --json
python -m nhydb.query shop.nhydb --realm 1 --eq "status=paid" --return count
```

Befehle: `to-n`, `from-n`, `layout`, `layout-check`, `layout-plan`,
`evolve-layout`, `route`, `describe`, `logical-address`, `occupied`,
`stats`, `health`, `get`, `address`, `cell`, `cell-n`, `find`, `verify`,
`rebuild-indexes`, `repair`, `restore`, `compact`. Die CLI verwendet die Standardorte der
Begleitdateien oder die Konfigurationsdatei. Lesebefehle öffnen die
Datenbank mit `readonly=True` und ändern die Datei nie; `verify` funktioniert auch
mit beschädigten Dateien. `repair` ist ein Probelauf, sofern nicht `--apply` angegeben ist
(`--allow-data-loss`, `--keep-uncommitted` wie bei `repair_file`). Exit-Codes:
`0` Erfolg, `1` eine Prüfung hat Probleme gefunden, `2` Bedienfehler oder abgelehnte
Operation, `3` Fehler (Meldung auf stderr, kein Traceback). In der JSON-Ausgabe
erscheinen `bytes` als `{"$bytes_b64": "…"}`. Dieselben Aufrufe aus Python:

In [12]:
import json
import os
import subprocess
import sys
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "cli.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "shop")
    db.create_shape(1, 1, "order", [{"v": 1, "name": "id", "type": "varuint"},
                                    {"v": 2, "name": "status", "type": "utf8"}])
    db.create_classic_layout(1, 1, "orders", shape_id=1)
    db.put_dense_batch(1, 1, [[1, "paid"], [2, "new"], [3, "paid"]])


def run(*args):
    return subprocess.run([sys.executable, *args], capture_output=True, text=True, check=True).stdout


print(json.loads(run("-m", "nhydb", "to-n", "--realm", "1", "--u", "11", "--v", "13", "--lane", "NR", "--json"))["N"])
print(json.loads(run("-m", "nhydb", "get", path, "u2,v2", "--realm", "1", "--mode", "cell", "--json"))["value"])
out = json.loads(run("-m", "nhydb.query", path, "--sql", "SELECT COUNT(*) AS n FROM orders WHERE status = ?",
                     "--params", '["paid"]', "--json"))
print(out["rows"])

589
new
[{'n': 2}]


*Erwartete Ausgabe (aus dem Cookbook):*

```text
589
new
[{'n': 2}]
```